In [52]:
import pandas as pd
import re
import tqdm

In [53]:
df = pd.read_csv('../news/final_df/cleaned_final_df.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13209 entries, 0 to 13208
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   title                   13209 non-null  str    
 1   outlet                  13209 non-null  str    
 2   date                    13209 non-null  str    
 3   authors                 13209 non-null  str    
 4   body                    13209 non-null  str    
 5   word_count              13209 non-null  int64  
 6   ai_related              13209 non-null  str    
 7   matched_keywords_title  13209 non-null  str    
 8   matched_keywords_body   13209 non-null  str    
 9   matched_keywords_all    13209 non-null  str    
 10  n_hits_title_total      13209 non-null  int64  
 11  n_hits_body_total       13209 non-null  int64  
 12  company_hits            13209 non-null  str    
 13  processed               13209 non-null  str    
 14  nouns                   13209 non-null  str    
 

In [54]:
keywords = (
    "AI Act|AI-verordening|Europese AI-wet|Europese AI-agenda|nationale AI-agenda|"
    "AI-wet|Europese AI-verordening|Europese AI-regels|nieuwe AI-wet|nieuwe AI-wetgeving|esprit|eureka"
)

# Convert into list
keywords = keywords.strip().split('|')
set_ai_words = {k for k in keywords if k.strip()}


In [55]:
_AI_PAT = re.compile(
    r"\b(" + "|".join(k.lower() for k in set_ai_words) + r")\b",
    re.IGNORECASE
)
# --- 3) Classification ---
def ai_classification(df, title_col='title', body_col='body'):

    labels, matched_title, matched_body, matched_all = [], [], [], []
    n_hits_title_total, n_hits_body_total = [], []

    for _, row in tqdm.tqdm(df.iterrows(), total=df.shape[0]):
        title = row[title_col] if pd.notna(row[title_col]) else ""
        body  = row[body_col]  if pd.notna(row[body_col])  else ""
 
        # regex matches
        title_matches = _AI_PAT.findall(str(title))
        body_matches  = _AI_PAT.findall(str(body))

        # decision rule
        title_match = len(title_matches) >= 1
        body_match  = len(body_matches)  >= 2
        label = "yes" if (title_match or body_match) else "no"

     
        labels.append(label)
        matched_title.append(sorted(set(m.lower() for m in title_matches)))
        matched_body.append(sorted(set(m.lower() for m in body_matches)))
        matched_all.append(sorted(set(m.lower() for m in (title_matches + body_matches))))
        n_hits_title_total.append(len(title_matches))
        n_hits_body_total.append(len(body_matches))

    # write back
    df['Jessy'] = labels
    # df['matched_keywords_title'] = matched_title
    # df['matched_keywords_body']  = matched_body
    df['Jessy_keywords']   = matched_all
    # df['n_hits_title_total'] = n_hits_title_total
    # df['n_hits_body_total']  = n_hits_body_total

    return df


In [56]:

# Apply classification (this modifies df and also drops weiwei rows)
df = ai_classification(df)

print(df['Jessy'].value_counts())

100%|██████████| 13209/13209 [00:03<00:00, 3915.43it/s]

Jessy
no     13155
yes       54
Name: count, dtype: int64


In [57]:
# drop rows where Jessy is 'no'
df = df[df['Jessy'] == 'yes'].copy()

In [58]:
df.to_csv('subset_jessy.csv', index=False)

In [59]:
jessy = pd.read_csv('subset_jessy.csv')


In [60]:
jessy.drop(columns=['ai_related', 'matched_keywords_title', 'matched_keywords_body', 'n_hits_total', 'n_hits_body_total', 'processed', 'nouns', 'adjectives', 'verbs', 'adverbs', 'date_parsed', 'year', 'topic', 'probability', 'topic_norm', 'topic_label_nl', 'Jessy', 'topic_meta_nl', 'topic_label', 'topic_meta', 'ai_words', 'n_hits_title_total'], inplace=True, errors='ignore')
jessy.rename(columns={'matched_keywords_all': 'AI_keywords', 'company_hits': 'company_keywords'}, inplace=True)
jessy.head()

,title,outlet,date,authors,body,word_count,AI_keywords,company_keywords,Jessy_keywords
0,‘Regering moet veel meer investeren in kunstma...,VK,2019-07-18,Laurens Verhagen,Studenten kunstmatige intelligentie (AI) vinde...,348,"['ai', 'artificial intelligence', 'kunstmatige...",['google'],['nationale ai-agenda']
1,Salons van melancholie en tederheid De roman a...,GA,2020-04-09 00:00:00,Margot Dijkgraaf,Sinds de literaire salons van Germaine de Staë...,3887,"['algoritme', 'kunstmatige intelligentie']",[],['esprit']
2,Professor Christian Nijhuis traint slimme mole...,AD,2022-12-28,Frank Timmers,ENSCHEDE - Een met het blote oog niet waarneem...,1049,"['artificiële intelligentie', 'kunstmatige int...",['asml'],['eureka']
3,Goedkopere zorg door technologie? Het wordt ju...,AD,2022-03-28,Bert van den Hoogen,Jan-Luuk Hoff uit Gouda onderzoekt hoe de opko...,999,"['algoritme', 'algoritmen', 'algoritmes']",[],['eureka']
4,EU-wet kan misbruik van ChatGPT niet voorkomen...,TR,2023-07-04,WASIMA KHAN,"Een nieuwe Europese wet, de AI Act, moet ervoo...",215,"['ai', 'algoritmes', 'chatgpt', 'kunstmatige i...",[],['ai act']


In [61]:
jessy.columns

Index(['title', 'outlet', 'date', 'authors', 'body', 'word_count',
       'AI_keywords', 'company_keywords', 'Jessy_keywords'],
      dtype='str')